# DeployStega Colab Full Run

This notebook pulls the latest DeployStega code, sets the Azure OpenAI runtime from Colab secrets, runs a token-binning smoke generation, then runs full covert trace generation and all adversarial evaluations including BERT.

**Data requirement:** the original experiment inputs must be present in Google Drive under `MyDrive/DeployStega_data/` as either folders or tarballs:

- `benign_traces/` or `benign_traces.tgz`
- `secrets/` or `secrets.tgz`
- `behavior_priors.json`
- `experiment_manifest.json`


In [ ]:
# Runtime setup
from google.colab import drive, userdata
import os, pathlib, shutil, subprocess, textwrap

drive.mount('/content/drive')

REPO_URL = 'https://github.com/TrishaSrikanth-459/DeployStega.git'
WORKDIR = pathlib.Path('/content/DeployStega')
DATA_DIR = pathlib.Path('/content/drive/MyDrive/DeployStega_data')

# Prefer Colab Secrets. If a secret is absent, the value remains unset and the runner will stop clearly.
secret_names = [
    'AZURE_OPENAI_API_KEY',
    'AZURE_OPENAI_ENDPOINT',
    'AZURE_OPENAI_API_VERSION',
    'AZURE_OPENAI_DEPLOYMENT_NAME',
    'AZURE_OPENAI_DEPLOYMENT',
]
for name in secret_names:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value

# Deployment fallback: code accepts either variable.
if os.environ.get('AZURE_OPENAI_DEPLOYMENT_NAME') and not os.environ.get('AZURE_OPENAI_DEPLOYMENT'):
    os.environ['AZURE_OPENAI_DEPLOYMENT'] = os.environ['AZURE_OPENAI_DEPLOYMENT_NAME']

# If you have not added Colab Secrets yet, uncomment and fill these locally in Colab only.
# os.environ['AZURE_OPENAI_DEPLOYMENT_NAME'] = 'gpt-5.2-chat'
# os.environ['AZURE_OPENAI_DEPLOYMENT'] = 'gpt-5.2-chat'
# os.environ['AZURE_OPENAI_API_VERSION'] = '2024-02-01'
# os.environ['AZURE_OPENAI_ENDPOINT'] = 'https://madhavanandmenon-6796-resource.cognitiveservices.azure.com/openai/responses?api-version=2025-04-01-preview'
# os.environ['AZURE_OPENAI_API_KEY'] = 'PASTE_IN_COLAB_ONLY'

required = [
    'AZURE_OPENAI_API_KEY',
    'AZURE_OPENAI_ENDPOINT',
    'AZURE_OPENAI_API_VERSION',
]
if not (os.environ.get('AZURE_OPENAI_DEPLOYMENT') or os.environ.get('AZURE_OPENAI_DEPLOYMENT_NAME')):
    required.append('AZURE_OPENAI_DEPLOYMENT_NAME')
missing = [name for name in required if not os.environ.get(name)]
print('Deployment:', os.environ.get('AZURE_OPENAI_DEPLOYMENT') or os.environ.get('AZURE_OPENAI_DEPLOYMENT_NAME'))
print('API version:', os.environ.get('AZURE_OPENAI_API_VERSION'))
print('Endpoint set:', bool(os.environ.get('AZURE_OPENAI_ENDPOINT')))
print('API key set:', bool(os.environ.get('AZURE_OPENAI_API_KEY')))
if missing:
    raise RuntimeError('Missing Colab Secrets/environment variables: ' + ', '.join(missing))


In [ ]:
# Pull latest code and install dependencies
import subprocess, pathlib, os, shutil
if WORKDIR.exists():
    subprocess.run(['git', '-C', str(WORKDIR), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)

subprocess.run(['python', '-m', 'pip', 'install', '-q', 'openai', 'scikit-learn', 'matplotlib', 'joblib', 'tqdm', 'transformers', 'torch', 'numpy', 'scipy'], check=True)
print(subprocess.check_output(['git', '-C', str(WORKDIR), 'rev-parse', '--short', 'HEAD']).decode().strip())


In [ ]:
# Restore experiment data from Google Drive
import tarfile, shutil, pathlib, os

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive')
DATA_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [
    DATA_DIR,
    DRIVE_ROOT,
    DRIVE_ROOT / 'DeployStega',
    DRIVE_ROOT / 'DeployStega_data',
]

def first_existing(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

def find_folder_or_archive(name):
    folder_candidates = [root / name for root in SEARCH_ROOTS]
    folder = first_existing(folder_candidates)
    if folder:
        return ('folder', folder)

    archive_candidates = []
    for root in SEARCH_ROOTS:
        archive_candidates.extend([root / f'{name}.tgz', root / f'{name}.tar.gz'])
    archive = first_existing(archive_candidates)
    if archive:
        return ('archive', archive)

    # One shallow recursive pass catches common manual uploads without scanning all Drive forever.
    for pattern in [f'*/{name}', f'*/{name}.tgz', f'*/{name}.tar.gz']:
        matches = sorted(DRIVE_ROOT.glob(pattern))
        if matches:
            found = matches[0]
            return ('folder' if found.is_dir() else 'archive', found)
    raise FileNotFoundError(
        f'Missing {name}. Expected {name}/, {name}.tgz, or {name}.tar.gz under one of: '
        + ', '.join(str(root) for root in SEARCH_ROOTS)
    )

def copy_or_extract(name, dest):
    dest = pathlib.Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    kind, source = find_folder_or_archive(name)
    print(f'Using {name} from {source}')
    dest.parent.mkdir(parents=True, exist_ok=True)
    if kind == 'folder':
        shutil.copytree(source, dest)
    else:
        with tarfile.open(source, 'r:*') as tf:
            tf.extractall(dest.parent)
        extracted = dest.parent / name
        if extracted != dest and extracted.exists() and not dest.exists():
            shutil.move(str(extracted), str(dest))

def copy_metadata(src_name, dst):
    candidates = [root / src_name for root in SEARCH_ROOTS]
    candidates.extend(sorted(DRIVE_ROOT.glob(f'*/{src_name}')))
    src = first_existing(candidates)
    if src is None:
        # These metadata files are versioned in the repo; using them is safer than blocking on Drive layout.
        repo_src = WORKDIR / src_name if src_name == 'behavior_priors.json' else WORKDIR / 'experiments/experiment_manifest.json'
        if repo_src.exists():
            print(f'Using repo default for {src_name}: {repo_src}')
            return
        raise FileNotFoundError(f'Missing {src_name} under Drive search roots')
    dst = pathlib.Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f'Using {src_name} from {src}')

copy_or_extract('benign_traces', WORKDIR / 'benign_traces')
copy_or_extract('secrets', WORKDIR / 'secrets')
copy_metadata('behavior_priors.json', WORKDIR / 'behavior_priors.json')
copy_metadata('experiment_manifest.json', WORKDIR / 'experiments/experiment_manifest.json')
print('benign traces:', len(list((WORKDIR/'benign_traces').glob('*.jsonl'))))
print('secrets:', len(list((WORKDIR/'secrets').glob('*.txt'))))
if not list((WORKDIR/'benign_traces').glob('*.jsonl')):
    raise RuntimeError('No benign trace JSONL files found after restore')
if not list((WORKDIR/'secrets').glob('*.txt')):
    raise RuntimeError('No secret TXT files found after restore')



In [ ]:
# Start the full run: smoke -> full generation -> cleanup -> all ablations including BERT
import subprocess, os, pathlib
os.chdir(WORKDIR)
cmd = [
    'python', 'scripts/run_colab_full_experiment.py',
    '--output-root', 'experiments/colab_full_run',
    '--target-traces', '4000',
    '--min-traces', '3500',
    '--workers', '1',
    '--smoke-traces', '25',
    '--semantic-smoke-max-auc', '0.98',
    '--semantic-smoke-max-accuracy', '0.95',
    '--bert-epochs', '3',
    '--bert-max-samples', '10000',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Package results back to Google Drive
import subprocess, pathlib, time, shutil
os.chdir(WORKDIR)
out_name = f'DeployStega_colab_full_results_{int(time.time())}.tgz'
subprocess.run(['tar', '-czf', out_name, 'experiments/colab_full_run'], check=True)
shutil.copy2(out_name, DATA_DIR / out_name)
print('Saved:', DATA_DIR / out_name)
